In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, upper

spark = SparkSession.builder \
    .appName("ETL_with_Repartition_Coalesce") \
    .getOrCreate()

spark

In [4]:
# Sample input path and output paths
input_path = "./data/input/customers.csv"  # adjust as needed

etl_output_path = "./data/output/customers_etl"  # parquet
repartition_output_path = "./data/output/customers_repartition"  # parquet
coalesce_output_path = "./data/output/customers_coalesce"  # parquet

In [14]:
# Basic ETL: read, clean, transform, write
df_raw = spark.read.option("header", "true").csv(input_path)
df_raw.show()

# Example transformations
df_clean = df_raw.select(
    trim(col("customer_id")).alias("customer_id"),
    upper(trim(col("first_name"))).alias("first_name"),
    upper(trim(col("last_name"))).alias("last_name"),
    trim(col("country")).alias("country"),
    col("signup_ts").cast("timestamp").alias("signup_ts")
)

# Filter out invalid rows
df_etl = df_clean.filter(col("customer_id").isNotNull())

# Write ETL result as parquet (default number of partitions)
# df_etl.write.mode("overwrite").parquet(etl_output_path)


# df_etl.printSchema()
# df_etl.columns
# df_etl.show(5, truncate=False)

+-----------+----------+---------+-------+-------------------+
|customer_id|first_name|last_name|country|          signup_ts|
+-----------+----------+---------+-------+-------------------+
|       C001|      John|      Doe|    USA|2024-01-05 10:15:00|
|       C002|       Ana|    Silva| Brazil|2024-02-11 09:00:00|
|       C003|       Wei|     Chen|  China|2024-03-22 14:45:00|
|       C004|      Sara|     Khan|    UAE|2024-04-01 08:30:00|
|       C005|     David|    Smith|    USA|2024-05-19 16:20:00|
|       C006|     Maria| Gonzalez|  Spain|2024-06-25 11:10:00|
|       C007|     Kenji|   Tanaka|  Japan|2024-07-03 13:55:00|
|       C008|      Lara|   Nasser|    UAE|2024-08-14 17:40:00|
|       C009|       Tom|      Lee|    USA|2024-09-09 12:05:00|
|       C010|     Aisha|      Ali|    UAE|2024-10-21 07:50:00|
+-----------+----------+---------+-------+-------------------+



In [17]:
# Repartition example: increase or redistribute partitions
# Typically used before wide operations or to improve parallelism
df_repartitioned = df_etl.repartition(8, col("country"))  # 8 partitions, partitioned by country

print("Originat  number of partitions:", df_etl.rdd.getNumPartitions())
print("Repartitioned number of partitions:", df_repartitioned.rdd.getNumPartitions())

# Write repartitioned data
df_repartitioned.write.mode("overwrite").parquet(repartition_output_path)

# df_repartitioned.write.mode("overwrite").csv(repartition_output_path)

Originat  number of partitions: 1
Repartitioned number of partitions: 8


In [23]:
# Coalesce example: reduce number of partitions (often before writing)
# Coalesce avoids full shuffle when reducing partitions
df_coalesced = df_etl.coalesce(2)  # reduce to 2 partitions

print("Coalesced number of partitions:", df_coalesced.rdd.getNumPartitions())

# Write coalesced data
df_coalesced.write.mode("overwrite").parquet(coalesce_output_path)

Coalesced number of partitions: 1


In [24]:
# Stop Spark session when done
spark.stop()